In [ ]:
!pip install requests beautifulsoup4 pandas


In [ ]:
# Instalamos Selenium y configuramos el entorno
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

# Configuramos Chrome sin interfaz (headless)
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from IPython.display import display

# 1. Scroll profundo para cargar todas las tarjetas Scroll para cargar toda la página
for _ in range(25):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

# 2. Obtener el HTML de la página
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
empresas = []

# 3. Buscar tarjetas de empresa seleccionas todas las tarjetas donde vive la info de cada empresa
tarjetas = soup.select('div[class^="relative flex w-full"]')
for tarjeta in tarjetas:
    nombre = tarjeta.select_one('span[class*="_coName"]')
    sede = tarjeta.select_one('span[class*="_coLocation"]')
    descripcion = tarjeta.select_one('span[class*="_coDescription"]')
    logo_tag = tarjeta.select_one('img')

    # Buscar enlace
    contenedor_a = tarjeta.find_parent('a')
    enlace = 'https://www.ycombinator.com' + contenedor_a['href'] if contenedor_a else ''

    # Buscar todos los <span class="pill"> y extraer texto datos de cada empresa
    pills = tarjeta.select('span.pill')
    batch = ''
    sectores = []

    for pill in pills:
        texto = pill.get_text(strip=True)
        if '20' in texto or 'Verano' in texto or 'Invierno' in texto:
            batch = texto
        else:
            sectores.append(texto)

    logo = logo_tag['src'] if logo_tag else ''

    empresas.append({
        'Nombre de la empresa': nombre.get_text(strip=True) if nombre else '',
        'Descripción': descripcion.get_text(strip=True) if descripcion else '',
        'Sede': sede.get_text(strip=True) if sede else '',
        'Batch': batch,
        'Sectores': ', '.join(sectores),
        'Enlace': enlace,
        'Logo': logo
    })

# 4. Guardar CSV los datos
df = pd.DataFrame(empresas)
df.to_csv('/content/empresas_ycombinator.csv', index=False)
print("✅ CSV actualizado con batch y sectores correctamente extraídos")
display(df)



✅ CSV actualizado con batch y sectores correctamente extraídos


,Nombre de la empresa,Descripción,Sede,Batch,Sectores,Enlace,Logo
0,DoorDash,Restaurant delivery.,"San Francisco, CA, USA",Summer 2013,"Consumer, Food and Beverage",https://www.ycombinator.com/companies/doordash,https://bookface-images.s3.amazonaws.com/small_logos/d13287c52acc96909f32342e85c26a33cfdac310.png
1,Airbnb,Book accommodations around the world.,"San Francisco, CA, USA",Winter 2009,"Consumer, Travel, Leisure and Tourism",https://www.ycombinator.com/companies/airbnb,https://bookface-images.s3.amazonaws.com/small_logos/3e9a0092bee2ccf926e650e59c06503ec6b9ee65.png
2,Coinbase,"Buy, sell, and manage cryptocurrencies.","San Francisco, CA, USA",Summer 2012,"Fintech, Banking and Exchange",https://www.ycombinator.com/companies/coinbase,https://bookface-images.s3.amazonaws.com/small_logos/d583cc2bc592cccd5ff68e81f8fce6bc48be8025.png
3,Instacart,Marketplace for grocery delivery and pickup,"San Francisco, CA, USA",Summer 2012,"Consumer, Food and Beverage",https://www.ycombinator.com/companies/instacart,https://bookface-images.s3.amazonaws.com/small_logos/9750fca21baaee75e035f1baaf58df8e2f5dcc67.png
4,Oklo,"Emission free, always on power from advanced fission power plants.","Santa Clara, CA, USA",Summer 2014,"Industrials, Energy",https://www.ycombinator.com/companies/oklo,https://bookface-images.s3.amazonaws.com/small_logos/7296c0e632938afdac098d754da89730bfc674b7.png
...,...,...,...,...,...,...,...
995,Evolvere BioSciences,Making Next-Generation Antibiotics that Outpace Bacterial Evolution,"Oxford, England, United Kingdom",Summer 2024,"Healthcare, Therapeutics",https://www.ycombinator.com/companies/evolvere-biosciences,https://bookface-images.s3.amazonaws.com/small_logos/dd3489b7e5a5f2d3f7cc49de65917a83134da0a3.png
996,Spur,Spur is your AI QA Engineer. Test your websites with natural language.,"San Francisco, CA, USA",Summer 2024,"B2B, Engineering, Product and Design",https://www.ycombinator.com/companies/spur,https://bookface-images.s3.amazonaws.com/small_logos/5b63801706e3874e4f1017433f3c6a3ce0dddbca.png
997,OddsView,The Bloomberg Terminal of Sports Betting,"New York, NY, USA",Winter 2024,Consumer,https://www.ycombinator.com/companies/oddsview,https://bookface-images.s3.amazonaws.com/small_logos/2d005af7cd2923d6f72a4eb0931818b5877bf89e.png
998,Silogy,An AI-powered test and debug platform for chip developers,"New York, NY, USA",Winter 2024,"B2B, Engineering, Product and Design",https://www.ycombinator.com/companies/silogy,https://bookface-images.s3.amazonaws.com/small_logos/01e853c98eeaa75f49024ede178b38c98d3e9a26.png


In [ ]:
from IPython.display import HTML

df_logo = df.copy()
df_logo['Logo'] = df_logo['Logo'].apply(lambda url: f'<a href="{url}" target="_blank"><img src="{url}" width="40"/></a>')

HTML(df_logo.to_html(escape=False))


,Nombre de la empresa,Descripción,Sede,Batch,Sectores,Enlace,Logo
0,DoorDash,Restaurant delivery.,"San Francisco, CA, USA",Summer 2013,"Consumer, Food and Beverage",https://www.ycombinator.com/companies/doordash,
1,Airbnb,Book accommodations around the world.,"San Francisco, CA, USA",Winter 2009,"Consumer, Travel, Leisure and Tourism",https://www.ycombinator.com/companies/airbnb,
2,Coinbase,"Buy, sell, and manage cryptocurrencies.","San Francisco, CA, USA",Summer 2012,"Fintech, Banking and Exchange",https://www.ycombinator.com/companies/coinbase,
3,Instacart,Marketplace for grocery delivery and pickup,"San Francisco, CA, USA",Summer 2012,"Consumer, Food and Beverage",https://www.ycombinator.com/companies/instacart,
4,Oklo,"Emission free, always on power from advanced fission power plants.","Santa Clara, CA, USA",Summer 2014,"Industrials, Energy",https://www.ycombinator.com/companies/oklo,
5,Dropbox,Backup and share files in the cloud.,"San Francisco, CA, USA",Summer 2007,"B2B, Productivity",https://www.ycombinator.com/companies/dropbox,
6,GitLab,A complete DevOps platform delivered as a single application.,"San Francisco, CA, USA",Winter 2015,"B2B, Engineering, Product and Design",https://www.ycombinator.com/companies/gitlab,
7,Rigetti Computing,Quantum coherent supercomputing.,"Berkeley, CA, USA",Summer 2014,"Industrials, Manufacturing and Robotics",https://www.ycombinator.com/companies/rigetti-computing,
8,Matterport,Turn physical objects and environments into 3D models in seconds.,"Sunnyvale, CA, USA",Winter 2012,"Consumer, Virtual and Augmented Reality",https://www.ycombinator.com/companies/matterport,
9,Amplitude,Digital Analytics Platform,"San Francisco, CA, USA",Winter 2012,"B2B, Analytics",https://www.ycombinator.com/companies/amplitude,


In [ ]:
# Ver cuántas empresas tienen sector o batch vacío
print("Sin Batch:", df['Batch'].eq('').sum())
print("Sin Sector:", df['Sectores'].eq('').sum())


Sin Batch: 1
Sin Sector: 0


In [ ]:
df['Sectores'].str.split(', ').explode().value_counts().head(10)


,count
Sectores,
B2B,571
Engineering,135
Product and Design,135
Consumer,112
Healthcare,98
Industrials,82
Fintech,70
Infrastructure,53
Productivity,40


resumen por sectores

In [ ]:
!pip install plotly
import plotly.express as px


In [ ]:
batch_counts = df['Batch'].value_counts().reset_index()
batch_counts.columns = ['Batch', 'Empresas']

fig = px.bar(batch_counts.sort_values('Batch'),
             x='Batch', y='Empresas',
             title='Empresas por batch (promoción)',
             labels={'Batch': 'Batch', 'Empresas': 'Número de empresas'},
             template='plotly_white',
             color='Empresas')
fig.update_layout(xaxis_tickangle=-45)
fig.show()


Este gráfico muestra cuántas empresas hay en cada batch (es decir, cada promoción o generación como “Verano 2013”, “Invierno 2022”, etc.). Te ayuda a ver en qué épocas salieron más startups —por ejemplo, si hubo una explosión de proyectos en cierto año.

In [ ]:
sectores = df['Sectores'].dropna().str.split(', ').explode()
top_sectores = sectores.value_counts().head(10).reset_index()
top_sectores.columns = ['Sector', 'Empresas']

fig = px.bar(top_sectores,
             x='Empresas', y='Sector',
             orientation='h',
             title='Top 10 sectores',
             template='plotly_dark',
             color='Empresas')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


Aquí aparecen los sectores o industrias con más presencia entre las startups (como “Alimentos y bebidas”, “Salud”, “Software”…). Te da una idea clara de qué tipo de empresas dominan la lista: ¿hay más tecnológicas, de medicina, de energía, etc.?

In [ ]:
sedes = df['Sede'].value_counts().head(6).reset_index()
sedes.columns = ['Sede', 'Empresas']

fig = px.pie(sedes, names='Sede', values='Empresas',
             title='Distribución por sede (top 6)',
             template='seaborn',
             hole=0.4)
fig.show()


Este gráfico circular (de pastel) que muestra las 5 ubicaciones más comunes donde están registradas las empresas. Por ejemplo, si la mayoría están en San Francisco o Los Ángeles, eso se refleja en los porcentajes.

In [ ]:
df.to_csv('/content/empresas_ycombinator.csv', index=False)


In [6]:
# 1. CONFIGURA TU IDENTIDAD (obligatorio para hacer commits)
!git config --global user.email "senoy99@hotmail.com"
!git config --global user.name "senoy99"

# 2. INICIALIZA EL REPOSITORIO
!git init

# 3. CREA README O ARCHIVO BASE
!echo "# Web-Scraping-Startups" > README.md

# 4. AÑADE Y CONFIRMA EL ARCHIVO
!git add README.md
!git commit -m "Primer commit con README"

# 5. CREA RAMA 'main' Y AJÚSTALA COMO PRINCIPAL
!git branch -M main

# 6. ELIMINA REMOTO SI YA EXISTÍA
!git remote remove origin

# 7. AGREGA EL REMOTO USANDO SSH
!git remote add origin git@github.com:senoy99/Web-Scraping-Startups.git

# 8. AGREGA HOST CLAVE PARA VERIFICACIÓN SSH
!mkdir -p ~/.ssh
!ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts

# 9. HAZ PUSH FINAL A LA RAMA 'main'
!git push -u origin main


Reinitialized existing Git repository in /content/.git/
On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
# github.com:22 SSH-2.0-44082949
git@github.com: Permission denied (publickey).
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


In [7]:
# 1. GENERAR CLAVE SSH (si no la tienes)
!ssh-keygen -t ed25519 -C "senoy99@hotmail.com" -f ~/.ssh/id_ed25519 -N ""

# 2. INICIAR EL AGENTE SSH Y AGREGAR LA CLAVE
!eval "$(ssh-agent -s)"
!ssh-add ~/.ssh/id_ed25519

# 3. MOSTRAR LA CLAVE PÚBLICA PARA COPIARLA
!cat ~/.ssh/id_ed25519.pub


Generating public/private ed25519 key pair.
Your identification has been saved in /root/.ssh/id_ed25519
Your public key has been saved in /root/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:6S+HmTomQHem2lxz9xZkYJIN+jzICfVHq7W8+NpZEX0 senoy99@hotmail.com
The key's randomart image is:
+--[ED25519 256]--+
|      . .+.      |
|     . oo.+. .   |
|    . . .o+.. . E|
|  . .oo= * .o. . |
| . . ++ S oo.    |
|  . . o..o....   |
|   = . oo=....   |
|  . + o =+.oo    |
|     o.o.+=.     |
+----[SHA256]-----+
Agent pid 2773
Could not open a connection to your authentication agent.
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIPIS9WIysysbWGHKiVmrqSyR1R2URalwYo0GmxXtDFT6 senoy99@hotmail.com


In [15]:
# 1️⃣ CONFIGURA TU IDENTIDAD
!git config --global user.name "senoy99"
!git config --global user.email "senoy99@hotmail.com"

# 2️⃣ CLONA TU REPOSITORIO USANDO TU TOKEN
# Reemplaza TU_TOKEN con tu token personal de GitHub (ej. ghp_xxxxx...)
!git clone https://TU_TOKEN@github.com/senoy99/Web-Scraping-Startups.git

# 3️⃣ COPIA TU NOTEBOOK AL REPOSITORIO CLONADO
# Asegúrate de que el nombre del notebook sea correcto
!cp "/content/Copia de Web Scraping Startups.ipynb" Web-Scraping-Startups/

# 4️⃣ CAMBIA AL DIRECTORIO DEL REPOSITORIO Y CREA LA RAMA
%cd Web-Scraping-Startups
!git checkout -b desarrollo

# 5️⃣ HAZ COMMIT Y PUSH A LA NUEVA RAMA
!git add "Copia de Web Scraping Startups.ipynb"
!git commit -m "Subiendo notebook a rama desarrollo"
!git push origin desarrollo



Cloning into 'Web-Scraping-Startups'...
cp: cannot stat '/content/Copia de Web Scraping Startups.ipynb': No such file or directory
/content/Web-Scraping-Startups
Switched to a new branch 'desarrollo'
fatal: pathspec 'Copia de Web Scraping Startups.ipynb' did not match any files
On branch desarrollo

Initial commit

nothing to commit (create/copy files and use "git add" to track)
error: src refspec desarrollo does not match any
error: failed to push some refs to 'https://github.com/senoy99/Web-Scraping-Startups.git'


In [16]:
!ls /content


README.md  sample_data	Web-Scraping-Startups


In [17]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [18]:
!find /content/drive -type f -name "*.ipynb"


/content/drive/MyDrive/Colab Notebooks/Copia de Web Scraping Startups.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/01.numeros_operadores_variables.ipynb
/content/drive/MyDrive/Colab Notebooks/03.strings_tuples_lists.ipynb
/content/drive/MyDrive/Colab Notebooks/Copia de 03.strings_tuples_lists.ipynb
/content/drive/MyDrive/Colab Notebooks/04 (1).bucles_for_while.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/04.bucles_for_while.ipynb
/content/drive/MyDrive/Colab Notebooks/05.condicionales_casting.ipynb
/content/drive/MyDrive/Colab Notebooks/06.sets_dicts (1).ipynb
/content/drive/MyDrive/Colab Notebooks/07 (1).random.ipynb
/content/drive/MyDrive/Colab Notebooks/06.sets_dicts.ipynb
/content/drive/MyDrive/Colab Notebooks/07.random.ipynb
/content/drive/MyDrive/Colab Notebooks/Copia de 07.random.ipynb
/content/drive/MyDrive/Colab No